# Part 3 baseline (non-leading partner): Shreya Akotiya

The brief asks the partner who did not lead Part 3 to implement and run the complete pipeline, at least as a working
baseline. `part3_agents/simple_pipeline.py` calls the team's five agents (Planner → Vision → Prompt Engineering →
Image Generation → Critique) once each, in a straight line, with **no revise loop and no orchestration layer**.
This notebook runs it on one scenario of each type from `part3_agents/eval/scenarios.csv`.

**How to run:** Colab, Runtime → Change runtime type → **T4 GPU**. Colab secrets: `ANTHROPIC_API_KEY` and
`GITHUB_TOKEN`. Then Runtime → **Run all** (about 10–15 minutes, most of it downloading the models).

In [ ]:
# Cell 1 - run information
import os, subprocess, sys, datetime, platform
REPO, REF, WORK = "sakotiya/data266-lab2-team32", "main", "/content/data266-lab2-team32"
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
if not os.path.isdir(WORK):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "-q", "-b", REF, f"https://{token}@github.com/{REPO}.git", WORK], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("run by: Shreya Akotiya (Part 3 baseline, non-leading partner)")
print("date:", datetime.datetime.now().isoformat(timespec="minutes"), "| code:", REF, commit)
print("python:", platform.python_version(), "| GPU:", gpu or "NONE - switch the runtime to T4")

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -q -y torchao   # Colab's torchao 0.10 makes peft refuse to load the LoRA
os.environ["P3_LORA"] = "sakotiya/data266-team32-naruto-run2-lora-r32"   # same adapter as the team's Part 3 runs
from part3_agents.src import models
print({"vlm": models.VLM, "lora": models.LORA, "device": models.DEVICE})

## Run the simple pipeline on 5 scenarios (one per type)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from part3_agents.simple_pipeline import run_simple_pipeline

scen = pd.read_csv("part3_agents/eval/scenarios.csv")
picked = scen[scen.scenario_id.isin(["S01", "S04", "S07", "S10", "S15"])]
rows, outputs = [], {}
for _, r in picked.iterrows():
    print("=" * 70, f"\n{r.scenario_id} ({r.type}): {r.request}")
    try:
        res = run_simple_pipeline(f"part3_agents/test_images/{r.image}", r.request)
        err = ""
    except Exception as e:                    # the simple version has no failure handling; record it
        res, err = {}, f"{type(e).__name__}: {e}"
        print("  ERROR:", err)
    route = res.get("plan", {}).get("route")
    crit = res.get("critique", {})
    rows.append({"scenario_id": r.scenario_id, "type": r.type, "expected_route": r.expected_route,
                 "route": route, "route_correct": route == r.expected_route,
                 "critic_decision": crit.get("decision"), "clip_score": crit.get("clip_score"),
                 "seconds": round(res.get("seconds", float("nan")), 1), "message": res.get("message", err)[:120]})
    if res.get("final_image") is not None:
        outputs[r.scenario_id] = (r.image, res["final_image"])
results = pd.DataFrame(rows)
os.makedirs("part3_agents/metrics", exist_ok=True)
results.to_csv("part3_agents/metrics/simple_pipeline_results.csv", index=False)
results

In [ ]:
fig, axes = plt.subplots(len(outputs), 2, figsize=(8, 4 * len(outputs)), squeeze=False)
for row, (sid, (src, out)) in enumerate(outputs.items()):
    axes[row, 0].imshow(Image.open(f"part3_agents/test_images/{src}")); axes[row, 0].set_title(f"{sid} input")
    axes[row, 1].imshow(out); axes[row, 1].set_title(f"{sid} simple pipeline output")
    for ax in axes[row]: ax.axis("off")
plt.tight_layout(); plt.show()
print("routes correct:", int(results.route_correct.sum()), "/", len(results),
      "| median s/request:", results.seconds.median())

## Compared with the team's final system
The simple version shows the five agents working in order. What it does not have, and the team's `pipeline.run`
adds: the revise loop (at most 2 revisions with critic feedback and strength adjustment), one JSON trace per request,
and failure handling that turns an exception in any agent into a clear message.

Save: File → Download → `.ipynb`, and download `part3_agents/metrics/simple_pipeline_results.csv` from the file panel.